In [1]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path, MINIO_SPARK_ENDPOINT

spark = create_spark_session(
    "NYC Building Risk - Silver 311 Test"
)

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_SPARK_ENDPOINT)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 18:35:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/01 18:35:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.4.0
MinIO endpoint: http://host.docker.internal:9001


In [2]:
spark.stop()

In [3]:
spark = create_spark_session(
    "NYC Building Risk - Silver 311 Test"
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 3.4.0
Spark UI: http://ff7529cf1dd6:4040


In [4]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path, MINIO_SPARK_ENDPOINT

spark = create_spark_session(
    "NYC Building Risk - Silver 311 Test"
)

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_SPARK_ENDPOINT)

Spark version: 3.4.0
MinIO endpoint: http://host.docker.internal:9001


In [5]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path, MINIO_SPARK_ENDPOINT

spark = create_spark_session(
    "NYC Building Risk - Silver 311 Test"
)

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_SPARK_ENDPOINT)

Spark version: 3.4.0
MinIO endpoint: http://host.docker.internal:9001


In [6]:
bronze_file = minio_path(
    "bronze/311/"
    "year=2026/"
    "month=08/"
    "day=18/"
    "backfill/"
    "page_00001.json"
)

print("Reading:")
print(bronze_file)

Reading:
s3a://nyc-building-risk/bronze/311/year=2026/month=08/day=18/backfill/page_00001.json


In [7]:
df = (
    spark.read
    .option("multiline", "true")
    .json(bronze_file)
)

print("Rows:", df.count())

26/09/01 18:46:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Rows: 1000


In [8]:
df.printSchema()

root
 |-- address_type: string (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- city: string (nullable = true)
 |-- closed_date: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- open_data_channel_type: string (nullable =

In [9]:
df.select(
    "unique_key",
    "created_date",
    "agency",
    "complaint_type",
    "incident_address",
    "bbl",
    "borough",
    "latitude",
    "longitude"
).show(
    10,
    truncate=False
)

+----------+-----------------------+------+--------------------+----------------------+----------+--------+------------------+------------------+
|unique_key|created_date           |agency|complaint_type      |incident_address      |bbl       |borough |latitude          |longitude         |
+----------+-----------------------+------+--------------------+----------------------+----------+--------+------------------+------------------+
|70105322  |2026-08-18T00:02:36.000|HPD   |DOOR/WINDOW         |1084 NEW YORK AVENUE  |3049170006|BROOKLYN|40.64660340264543 |-73.94622758784209|
|70109629  |2026-08-18T00:04:34.000|HPD   |WATER LEAK          |1759 MONTGOMERY AVENUE|2028770458|BRONX   |40.85103455926995 |-73.91769461216724|
|70111230  |2026-08-18T00:04:34.000|HPD   |SAFETY              |1759 MONTGOMERY AVENUE|2028770458|BRONX   |40.85103455926995 |-73.91769461216724|
|70112723  |2026-08-18T00:04:34.000|HPD   |UNSANITARY CONDITION|1759 MONTGOMERY AVENUE|2028770458|BRONX   |40.85103455926995

In [11]:
from pyspark.sql import functions as F


silver_test_df = (
    df

    # --------------------------------------------------
    # 1. Keep fields relevant for Building Risk
    # --------------------------------------------------
    .select(
        "unique_key",
        "created_date",
        "closed_date",
        "resolution_action_updated_date",

        "agency",
        "agency_name",

        "complaint_type",
        "descriptor",
        "descriptor_2",

        "status",

        "incident_address",
        "street_name",
        "incident_zip",
        "borough",
        "city",

        "bbl",

        "latitude",
        "longitude",

        "community_board",
        "council_district",

        "location_type",
        "open_data_channel_type"
    )


    # --------------------------------------------------
    # 2. Convert dates from string to timestamp
    # --------------------------------------------------
    .withColumn(
        "created_date",
        F.to_timestamp(
            "created_date",
            "yyyy-MM-dd'T'HH:mm:ss.SSS"
        )
    )

    .withColumn(
        "closed_date",
        F.to_timestamp(
            "closed_date",
            "yyyy-MM-dd'T'HH:mm:ss.SSS"
        )
    )

    .withColumn(
        "resolution_action_updated_date",
        F.to_timestamp(
            "resolution_action_updated_date",
            "yyyy-MM-dd'T'HH:mm:ss.SSS"
        )
    )


    # --------------------------------------------------
    # 3. Convert coordinates to numeric values
    # --------------------------------------------------
    .withColumn(
        "latitude",
        F.col("latitude").cast("double")
    )

    .withColumn(
        "longitude",
        F.col("longitude").cast("double")
    )


    # --------------------------------------------------
    # 4. Normalize text fields
    # --------------------------------------------------
    .withColumn(
        "agency",
        F.upper(F.trim(F.col("agency")))
    )

    .withColumn(
        "complaint_type",
        F.upper(F.trim(F.col("complaint_type")))
    )

    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )

    .withColumn(
        "borough",
        F.upper(F.trim(F.col("borough")))
    )

    .withColumn(
        "incident_address",
        F.upper(F.trim(F.col("incident_address")))
    )


    # --------------------------------------------------
    # 5. Normalize BBL
    # --------------------------------------------------
    .withColumn(
        "bbl",
        F.when(
            F.col("bbl").rlike("^[0-9]{10}$"),
            F.col("bbl")
        ).otherwise(
            F.lit(None)
        )
    )


    # --------------------------------------------------
    # 6. Create useful partition fields
    # --------------------------------------------------
    .withColumn(
        "created_year",
        F.year("created_date")
    )

    .withColumn(
        "created_month",
        F.month("created_date")
    )

    .withColumn(
        "created_day",
        F.dayofmonth("created_date")
    )


    # --------------------------------------------------
    # 7. Remove duplicate 311 requests
    # --------------------------------------------------
    .dropDuplicates(
        ["unique_key"]
    )
)

In [12]:
silver_test_df.printSchema()

root
 |-- unique_key: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- resolution_action_updated_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- status: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- city: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- open_data_channel_type: string (nullable = true)
 |-- created_year: integ

In [13]:
print("Bronze rows:", df.count())
print("Silver rows:", silver_test_df.count())

Bronze rows: 1000
Silver rows: 1000


In [14]:
from pyspark.sql import functions as F

print(
    "Missing unique_key:",
    silver_test_df
    .filter(F.col("unique_key").isNull())
    .count()
)

print(
    "Missing created_date:",
    silver_test_df
    .filter(F.col("created_date").isNull())
    .count()
)

print(
    "Missing BBL:",
    silver_test_df
    .filter(F.col("bbl").isNull())
    .count()
)

print(
    "Missing coordinates:",
    silver_test_df
    .filter(
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    .count()
)

Missing unique_key: 0
Missing created_date: 0
Missing BBL: 0
Missing coordinates: 0


In [15]:
spark.stop()

In [16]:
spark = create_spark_session(
    "NYC Building Risk - Silver 311 Validation"
)


In [17]:
silver_path = minio_path(
    "silver/nyc_311"
)

silver_df = (
    spark.read
    .parquet(silver_path)
)

print("Silver rows:", silver_df.count())

Silver rows: 885306


In [18]:
silver_df.groupBy(
    "created_year",
    "created_month"
).count().orderBy(
    "created_year",
    "created_month"
).show(
    50,
    truncate=False
)

+------------+-------------+------+
|created_year|created_month|count |
+------------+-------------+------+
|2025        |8            |9122  |
|2025        |9            |42631 |
|2025        |10           |74594 |
|2025        |11           |86247 |
|2025        |12           |105826|
|2026        |1            |129816|
|2026        |2            |107413|
|2026        |3            |74385 |
|2026        |4            |60093 |
|2026        |5            |52407 |
|2026        |6            |46775 |
|2026        |7            |54663 |
|2026        |8            |41334 |
+------------+-------------+------+

